In [1]:
### USE THIS TO TEST VOYAGE AI'S EMBEDDING MODELS

import pandas as pd
## -- 12/26/24 -- adding new Ferguson dataset from Marty
from dotenv import load_dotenv
import os
load_dotenv()


import pandas as pd

df_final = pd.read_csv(r'data/ferg_plus_gt_44k_size_updated.csv')
print('len of df: ', len(df_final))
#df.head(5)

len of df:  44000


In [2]:
## load this dataset to qdrant for testing
## note that this collection is a sparse filter

# CONNECT TO QDRANT
import os
import json
import qdrant_client
from qdrant_client.http.models import Filter, FieldCondition, MatchValue,  MatchAny, PointStruct, VectorParams, Distance
from qdrant_client import QdrantClient


ENDLESSFORMS_QDRANT_URL = os.getenv("ENDLESSFORMS_QDRANT_URL")
ENDLESSFORMS_TEST_CLUSTER_KEY = os.getenv("ENDLESSFORMS_TEST_CLUSTER_KEY")

qdrantclient = QdrantClient(
    url=ENDLESSFORMS_QDRANT_URL,
    api_key=ENDLESSFORMS_TEST_CLUSTER_KEY,
    timeout=3000
)

print(qdrantclient)

collections = qdrantclient.get_collections()
list(collections)[0][1]

[CollectionDescription(name='TEXAS_PIPE_EXP'),
 CollectionDescription(name='test_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='test_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_HYBRID'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_BAAI_bge-reranker-v2-m3'),
 CollectionDescription(name='DENSE_VECTOR_BENCHMARK_25K_DEC17'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15_2'),
 CollectionDescription(name='ENCODER-TEST-INTFLOAT-MULTILINGUAL-E5-BASE'),
 CollectionDescription(name='ENCODER_TEST_2024-12-20_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-26_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='ENCODER

In [4]:
# load embedding model, check size

from dotenv import load_dotenv
load_dotenv()

import voyageai
VOYAGE_AI = os.getenv("VOYAGE_AI")
vo = voyageai.Client(api_key=VOYAGE_AI)

text = "testing this embedding model"

result = vo.embed(text, model="voyage-3-large", input_type='query', output_dimension=2048)

v = result.embeddings[0]
print(v)
print('len of v:', len(v))

[-0.03724543750286102, -0.01513135526329279, -0.0023080252576619387, -0.01840396411716938, 0.01329333707690239, 0.03128515183925629, -0.01637493073940277, -0.03861503675580025, -0.012860582210123539, -0.03207140415906906, -0.03169730305671692, 0.028832877054810524, 0.0005278656608425081, -0.014918942004442215, -0.011083593592047691, -0.024208899587392807, 0.016874263063073158, 0.0273919478058815, -0.029344890266656876, -0.03449990227818489, 0.06186014786362648, -0.04124009609222412, 0.005425286013633013, -0.0055033559910953045, -0.01590571738779545, 0.05130917206406593, -0.0225856751203537, 0.03161487355828285, -0.006944682914763689, -0.027290495112538338, 0.019578583538532257, 0.03954078257083893, -0.04203902930021286, 0.029446342960000038, 0.01191105879843235, -0.010761801153421402, -0.005820788908749819, 0.021078167483210564, -0.0067290980368852615, -0.051258452236652374, -0.014498076401650906, 0.015581151470541954, 0.05844883620738983, -0.049597177654504776, 0.00019378850993234664,

In [5]:
def voyage_vectorizer(voyageclient,encoder,text):
    """use voyageai client to encode text"""
    result = voyageclient.embed(text, model=encoder, input_type='query', output_dimension=2048)
    return result.embeddings[0]

In [ ]:
# # #DELETE OLD COLLECTION
# COLLECTION_NAME = "DENSE_VECTOR_FERGUSON_DEC_26_OPENAI_small"

# collection_name = COLLECTION_NAME

# # Delete the collection
# response = qdrantclient.delete_collection(collection_name=collection_name)

# # Print the response
# print(response)

In [6]:

from qdrant_client import QdrantClient, models
from qdrant_client.http.models import VectorParams

COLLECTION_NAME = "FERGUSON_VOYAGEAI_LARGE_2048_JAN1"
encoder = "voyage-3-large"
# Create the collection
if not qdrantclient.collection_exists(COLLECTION_NAME):
    qdrantclient.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=2048, distance=Distance.COSINE, on_disk = True)
    )

collections = qdrantclient.get_collections()
list(collections)[0][1]

[CollectionDescription(name='TEXAS_PIPE_EXP'),
 CollectionDescription(name='test_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='test_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_HYBRID'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_BAAI_bge-reranker-v2-m3'),
 CollectionDescription(name='DENSE_VECTOR_BENCHMARK_25K_DEC17'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15_2'),
 CollectionDescription(name='ENCODER-TEST-INTFLOAT-MULTILINGUAL-E5-BASE'),
 CollectionDescription(name='ENCODER_TEST_2024-12-20_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-26_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='ENCODER

In [7]:
## clean data

from EnhancedDataCleanser import DataCleanser

cleanser = DataCleanser(
        default_uppercase=True,
        measurement_standardization=True,
        fraction_conversion=True
    )

df_clean = cleanser.clean_dataframe(df_final)

In [8]:
print('len of clean data: ', len(df_clean))

len of clean data:  44000


In [9]:
df_clean.columns

Index(['UNNAMED 0', 'DESCRIPTION', 'CATEGORY', 'TYPE', 'PRIMARY_SIZE',
       'REDUCING_SIZE', 'LENGTH', 'MATERIAL_NAME', 'MATERIAL_SPECIFICATION',
       'MATERIAL_GRADE', 'PRESSURE_CLASS', 'PRIMARY_SCHEDULE',
       'REDUCING_SCHEDULE', 'MANUFACTURING_PROCESS', 'END_FINISH',
       'END_CONNECTIONS', 'TRIM', 'MISCELLANEOUS', 'CONFIDENCE'],
      dtype='object')

In [ ]:

import numpy as np

# LOAD DATA
# this is for single pointstruct loading
# Initialize vectorizer

embedding_model = "voyage-3-large"

bad_data_points = []

"""input dataframe, output PointSturct"""
for i, row in df_clean.iterrows():
    payload = {
        "ID": str(i),
        "DESCRIPTION":str(row['DESCRIPTION']),
        "CATEGORY": str(row['CATEGORY']),
        "TYPE": str(row['TYPE']),
        "PRIMARY_SIZE": str(row['PRIMARY_SIZE']),
        "REDUCING_SIZE": str(row['REDUCING_SIZE']),
        "LENGTH": row['LENGTH'],
        "MATERIAL_NAME": str(row['MATERIAL_NAME']),
        "MATERIAL_SPECIFICATION": row['MATERIAL_SPECIFICATION'],
        "MATERIAL_GRADE": row['MATERIAL_GRADE'],
        "PRESSURE_CLASS": row['PRESSURE_CLASS'],
        "PRIMARY_SCHEDULE": row['PRIMARY_SCHEDULE'],
        "REDUCING_SCHEDULE": row['REDUCING_SCHEDULE'],
        "MANUFACTURING_PROCESS": row['MANUFACTURING_PROCESS'],
        "END_FINISH": row['END_FINISH'],
        "END_CONNECTIONS": row['END_CONNECTIONS'],
        "TRIM": row['TRIM'],
        "MISCELLANEOUS": row['MISCELLANEOUS'],
        "CONFIDENCE": row['CONFIDENCE'],
    }

    desc = row['DESCRIPTION']

    if isinstance(desc,str):

        if desc != np.nan or desc != 'nan':

            #v = get_encoded_embedding(desc,encoder=encoder)
            #v = open_ai_vectorizer(openaiclient, embedding_model, desc)
            v = voyage_vectorizer(vo,embedding_model,desc)

            operation_info = qdrantclient.upsert(
                collection_name=COLLECTION_NAME,
                points=[
                    models.PointStruct(
                        id=i,
                        payload=payload,  # Add any additional payload if necessary
                        vector=v
                    )
                ],
            )

            print(operation_info)

    else:
        print('got a bad data point!')
        bad_data_points.append(payload)

print('all files uploaded!')
print('bad datapoints:', bad_data_points)

In [ ]:
print(len(bad_data_points))